# Connectome Perturbation: GPU Null Models

This notebook runs the 230 remaining permutations of the degree and distance null models using **Kaggle's free T4/P100 GPUs** via `brian2genn`.

## Setup Instructions
1. In the Kaggle Notebook editor, look at the right sidebar.
2. Under **Accelerator**, choose **GPU T4 x2** (or P100).
3. Under **Settings -> Internet**, toggle Internet to **On** (very important!).
4. Under **Input**, click **Add Input** -> **Upload a Dataset**.
5. Upload your entire `Drosophila_Data` folder as a dataset.
6. Once uploaded, run the cells below sequentially.

In [ ]:
# 1. Copy the Read-Only Dataset into the Writable Working Directory
import shutil
import os

input_dir = "/kaggle/input"
datasets = [d for d in os.listdir(input_dir) if not d.startswith(".")]
print("Found datasets:", datasets)

if len(datasets) > 0:
    ds_name = datasets[0]  # Assuming your uploaded folder is the only dataset
    src = os.path.join(input_dir, ds_name)
    dst = "/kaggle/working/Drosophila_Data"
    
    if not os.path.exists(dst):
        print(f"Copying {src} to {dst} (this takes a minute for the large parquet files)...")
        shutil.copytree(src, dst)
        print("Copy complete!")
    else:
        print("Data already exists in /kaggle/working.")
        
    # Self-healing CD: Find the actual root directory (in case zipping created a nested folder)
    os.chdir(dst)
    repo_root = "."
    for root, dirs, files in os.walk("."):
        if "model.py" in files:
            repo_root = root
            break
            
    os.chdir(repo_root)
    print("Current directory correctly set to:", os.getcwd())
else:
    raise FileNotFoundError("No dataset found. Please upload your Drosophila_Data folder to Kaggle.")

In [ ]:
# 2. Install Brian2, GeNN backend, and exact dependencies
!pip install numpy==1.26.4 brian2==2.5.4 pandas pyarrow pyyaml scipy joblib brian2genn
!git clone https://github.com/genn-team/genn.git /kaggle/working/genn

In [ ]:
import os
import re

# Set GENN and CUDA Paths for the current Python session
os.environ["GENN_PATH"] = "/kaggle/working/genn"
os.environ["CUDA_PATH"] = "/usr/local/cuda"

# 3. Find and patch model.py for GPU acceleration
model_path = None
for root, dirs, files in os.walk("."):
    if "model.py" in files:
        model_path = os.path.join(root, "model.py")
        break

if model_path:
    print(f"Found model.py at: {model_path}")
    with open(model_path, "r") as f:
        content = f.read()

    patched_code = (
        "import os\n"
        "os.environ['GENN_PATH'] = '/kaggle/working/genn'\n"
        "os.environ['CUDA_PATH'] = '/usr/local/cuda'\n"
        "import brian2genn\n"
        "from brian2 import set_device, prefs\n"
        "prefs.devices.genn.path = '/kaggle/working/genn'\n"
        "prefs.devices.genn.cuda_backend.cuda_path = '/usr/local/cuda'\n"
        "set_device('genn')"
    )

    # Overwrite the file from scratch if it was already patched to avoid duplicate imports
    if "prefs.codegen.target = 'numpy'" in content:
        content = content.replace("prefs.codegen.target = 'numpy'", patched_code)
        with open(model_path, "w") as f:
            f.write(content)
        print("Patched model.py successfully with CUDA_PATH, GENN_PATH and GPU acceleration.")
    elif "CUDA_PATH" not in content:
        # If already patched with GeNN but missing CUDA, we do a regex replace to upgrade it
        content = re.sub(r"import os\nos\.environ\['GENN_PATH'\].*?set_device\('genn'\)", patched_code, content, flags=re.DOTALL)
        with open(model_path, "w") as f:
            f.write(content)
        print("Upgraded patched model.py to include CUDA_PATH.")
    else:
        print("model.py already properly configured.")
else:
    print("Could not locate model.py.")

In [ ]:
# 4. Run the Degree-Matched Nulls (Resumes automatically based on files in results/)
!python scripts/run_degree_matched_nulls.py --config configs/jo_ground_truth_n20.yaml --n-perms 30 --groups AN descending LO Kenyon_Cell motor

In [ ]:
# 5. Run the Distance-Matched Nulls
!python scripts/run_distance_matched_nulls.py --config configs/jo_ground_truth_n20.yaml --n-perms 30 --groups AN descending LO Kenyon_Cell motor

In [ ]:
# 6. Verify and Combine final results
!python scripts/verify_and_combine_nulls.py --results-dir results/jo_ground_truth_n20/

In [ ]:
# 7. Zip the results so you can download them directly from the Kaggle Output sidebar
!zip -r /kaggle/working/results_archive.zip results/jo_ground_truth_n20/